In [68]:
import optuna
from typing import Callable
import pandas as pd

SAVE_FILENAME = 'results.csv'

In [69]:
def pol(odd_coefs):
    def f(x: float):
        ans = 0
        summer = x
        x_sqr = x*x
        for coef in odd_coefs:
            ans += coef * summer
            summer *= x_sqr
        return ans
    return f

In [70]:
def repeat_func(f: Callable, times: int):
    def ans(x):
        result = x
        for _ in range(times):
            result = f(result)
        return result
    return ans

In [71]:
def sign_l2_loss(f: Callable, interval=[0,1], h=1e-5):
    assert len(interval) == 2, "Интервал это 2 числа."
    left, right = interval
    assert left < right    
    n_steps = int((right - left) / h)
    assert n_steps != 0, "Слишком маленький интервал по сравнению с шагом"
    try:
        ans = f(0) ** 2
        for i in range(1, n_steps):
            x = left + i * h
            ans += (f(x)-1) ** 2
        ans = ans / (right - left)
        return ans
    except OverflowError:
        return float('+inf')

In [72]:
def abstruct_objective(coef_len: int, max_x: float, n: int):
    def objective(trial: optuna.Trial):
        odd_coefs = []
        for i in range(coef_len):
            odd_coefs.append(
                trial.suggest_float(f"a{i}", -10, 10)
            )
        f = pol(odd_coefs)
        f = repeat_func(f, n)
        loss = sign_l2_loss(f, [0, max_x], max_x / 1e4)
        return loss
    return objective

In [73]:
def experiment(coef_len: int, max_x: float, n: int):
    sampler = optuna.samplers.CmaEsSampler()
    study = optuna.create_study(
        sampler=sampler,
        direction='minimize'
    )
    study.optimize(abstruct_objective(coef_len, max_x, n), n_jobs=-1, n_trials=1000)
    return (coef_len, max_x, n, study.best_value, study.best_params)


In [74]:
results = []
for coef_len in [3, 5, 7]:
    odd_coef_len = coef_len // 2 + 1
    for max_x in [1, 5, 10, 20]:
        for n in [5, 8]:
            results.append(
                experiment(odd_coef_len, max_x, n)
            )

[I 2025-12-02 05:25:45,271] A new study created in memory with name: no-name-6469a5f9-96a5-4d0d-9834-473428cedb9f
[I 2025-12-02 05:25:45,289] Trial 0 finished with value: 1.9457624163083915e+138 and parameters: {'a0': 8.851742336063225, 'a1': -0.8486433418471524}. Best is trial 0 with value: 1.9457624163083915e+138.
[I 2025-12-02 05:25:45,308] Trial 1 finished with value: 7.87628454628223e+75 and parameters: {'a0': -4.228478264981331, 'a1': 4.4255482649427424}. Best is trial 1 with value: 7.87628454628223e+75.
[I 2025-12-02 05:25:45,344] Trial 2 finished with value: 6.380332940130068e+158 and parameters: {'a0': -1.8657029224507973, 'a1': -3.294729208926568}. Best is trial 1 with value: 7.87628454628223e+75.
[I 2025-12-02 05:25:45,359] Trial 4 finished with value: 7.000821440523939e+185 and parameters: {'a0': 1.0396581650154353, 'a1': 5.10422773817856}. Best is trial 1 with value: 7.87628454628223e+75.
[I 2025-12-02 05:25:45,380] Trial 3 finished with value: 4.5260516970050185e+84 and p

In [75]:
results

[(2,
  1,
  5,
  141.56036809924203,
  {'a0': 1.9627752262133047, 'a1': -0.9745491650215143}),
 (2,
  1,
  8,
  22.69615435207973,
  {'a0': -1.9121488142689405, 'a1': 0.9088365197634989}),
 (2,
  5,
  5,
  204.83976695769383,
  {'a0': 0.9615467770216917, 'a1': -0.0321201869270098}),
 (2,
  5,
  8,
  14400.461011353105,
  {'a0': 2.6113451606183595, 'a1': -0.11910720115962192}),
 (2,
  10,
  5,
  118.79050700038044,
  {'a0': 0.8080260376536792, 'a1': -0.006529112300960094}),
 (2, 10, 8, inf, {'a0': -0.21436005930580926, 'a1': 6.2820635480254765}),
 (2,
  20,
  5,
  63.581060940437226,
  {'a0': 0.6922723314094537, 'a1': -0.0013890883829219547}),
 (2, 20, 8, inf, {'a0': 8.499259380507329, 'a1': 2.9685875543959526}),
 (3,
  1,
  5,
  15.834245269328113,
  {'a0': 3.100870203589764,
   'a1': -3.4236723302523187,
   'a2': 1.318381706386182}),
 (3,
  1,
  8,
  0.2090960605151581,
  {'a0': 3.0795121609710137,
   'a1': -3.3866365676626327,
   'a2': 1.306934570637413}),
 (3,
  5,
  5,
  inf,
  {'a

In [76]:
coef_counts, max_xs, NSs, losses, ds = zip(*results)
d_a1 = [d.get('a0', 0) for d in ds]
d_a3 = [d.get('a1', 0) for d in ds]
d_a5 = [d.get('a2', 0) for d in ds]
d_a7 = [d.get('a3', 0) for d in ds]

In [77]:
df = pd.DataFrame({
    "loss": losses,
    "coef_count": coef_counts,
    "max_x": max_xs,
    "NS_n": NSs,
    "a1": d_a1,
    "a3": d_a3,
    "a5": d_a5,
    "a7": d_a7,
})

In [79]:
df.sort_values('NS_n')

,loss,coef_count,max_x,NS_n,a1,a3,a5,a7
0,1.415604e+02,2,1,5,1.962775,-0.974549,0.000000,0.000000
2,2.048398e+02,2,5,5,0.961547,-0.032120,0.000000,0.000000
20,inf,4,10,5,-5.328946,6.897594,5.261717,-4.577225
4,1.187905e+02,2,10,5,0.808026,-0.006529,0.000000,0.000000
6,6.358106e+01,2,20,5,0.692272,-0.001389,0.000000,0.000000
18,inf,4,5,5,1.075091,-1.053909,-0.915769,-7.395404
8,1.583425e+01,3,1,5,3.100870,-3.423672,1.318382,0.000000
10,inf,3,5,5,-7.926128,-2.239759,9.823365,0.000000
22,inf,4,20,5,-8.256968,5.954262,0.283987,7.529548
12,inf,3,10,5,-9.573746,-3.832489,-8.405640,0.000000


In [82]:
df.to_csv(path_or_buf=SAVE_FILENAME, index=False)